# 12 – A Clean Test on New Matches

**Objective:** measure the final model on matches played after all of its data, which nobody has looked at while building it.

**Key results**
- **The score fell on new matches:** on 1,253 matches, the model reached 65.28% accuracy and a log loss of 0.6243, against 66.98% and 0.5929 in notebook 08. Elo and the bookmakers also scored worse in this period.
- **It only just beats Elo:** the gain in log loss is +0.0100, and the lower end of its interval is +0.00001. This passes the rule, but by so little that it isn't strong evidence.
- **It doesn't beat the market:** its log loss is 0.6221 against 0.6153 for the market average on 1,202 matches (gain −0.0068, 95% interval: −0.0165 to +0.0027).
- **No proven profit:** the betting rule at Bet365 returned +0.81% on 426 bets, and its interval is very wide (−11.66% to +13.91%).

**Data:** main-tour matches from June to November 2026, from TennisMyLife. `src/update_data.py` builds them, and earlier matches stay exactly as they were.

## 1. Rules fixed before looking at any result

| Choice | Decision |
|---|---|
| **Model** | The same logistic regression with 60 columns, trained once on main-tour matches from 2005 up to Sackmann's last matches. Nothing is refitted |
| **Test matches** | Main-tour matches that aren't in `tennis.db`, without Davis Cup or walkovers |
| **Benchmark** | Pinnacle has no prices in this period, so the main benchmark is the market average, as also used in notebook 08 |

**Three tests decide the outcome. Each passes only if the lower end of its 95% bootstrap interval is above zero.**

1. **Model against the market average:** log loss gain per match.
2. **Model against Elo alone:** log loss gain per match.
3. **Betting rule:** bet 1 unit when model probability × odds − 1 is at least 5%, at Bet365, with retirements refunded, as in notebook 08.

**Only reported:** accuracy, Bet365 and Betfair Exchange as other benchmarks, the betting rule at other prices, and the calibration slope.

**The seed rule can't be tested in its frozen form,** because it needs Pinnacle's prices. Section 6 tries it with other prices, which is only exploratory.

**Checks made before running:** which bookmakers have prices, and how many matches link to the odds. Neither looks at the model's predictions. Linking was improved for four players whose names are written differently.

In [1]:
import sqlite3
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")
sys.path.insert(0, "../src")
from tennis_domain_features import DOMAIN_GROUPS
from tennis_elo import win_probability
from tennis_features import FEATURE_GROUPS
from tennis_market import fair_probability, roi_summary, seed_rule_probability
from tennis_models import INTERACTIONS, add_interactions, fit_both_views, logistic_model, predict_winner_probability, score
from tennis_odds import link_odds, load_odds
from tennis_tml import names_for_odds_link

ORDER = ["Elo", "Ranking", "Serve and return", "Form", "Fatigue and fitness", "Experience",
         "Style and age", "Head to head", "Draw", "Home"]
COLUMNS = ([c for group in ORDER for c in FEATURE_GROUPS[group]]
           + list(INTERACTIONS) + DOMAIN_GROUPS["Tournament history"])
KEYS = ["tourney_id", "match_num"]

latest = sqlite3.connect(Path("../data/processed/tennis_latest.db"))
query = "SELECT * FROM final_features WHERE tourney_level <> 'D'"
old = add_interactions(pd.read_sql_query(query, sqlite3.connect(Path("../data/processed/tennis.db"))))
everything = add_interactions(pd.read_sql_query(query, latest))
merged = everything.merge(old[KEYS], on=KEYS, how="left", indicator=True)
test = merged[merged["_merge"] == "left_only"].drop(columns="_merge").reset_index(drop=True)
train = old[old["year"] >= 2005]

model = fit_both_views(logistic_model(), train[COLUMNS])
rank_model = fit_both_views(logistic_model(), train[["d_log_rank"]])
test["model_p"] = predict_winner_probability(model, test[COLUMNS])
test["elo_p"] = np.asarray(win_probability(test["d_elo_combined"], 0), dtype=float)
test["rank_p"] = predict_winner_probability(rank_model, test[["d_log_rank"]])
print(f"Columns: {len(COLUMNS)}. Training matches: {len(train):,}. Test matches: {len(test):,}")

Columns: 60. Training matches: 56,809. Test matches: 1,253


## 2. How good is the model on these matches?

In [2]:
def interval(values, rounds=10000, seed=0):
    """Mean of the values, with a 95% bootstrap interval from resampling matches."""
    values = np.asarray(values, dtype=float)
    rng = np.random.default_rng(seed)
    means = values[rng.integers(0, len(values), size=(rounds, len(values)))].mean(axis=1)
    low, high = np.percentile(means, [2.5, 97.5])
    return {"gain": round(float(values.mean()), 5), "low_95": round(float(low), 5), "high_95": round(float(high), 5), "passes": bool(low > 0)}


display(pd.DataFrame([{"method": "Final model", **score(test["model_p"])}, {"method": "Elo only", **score(test["elo_p"])},
                      {"method": "ATP ranking only", **score(test["rank_p"])}]))
elo_test = interval(np.log(test["model_p"]) - np.log(test["elo_p"]))
print("Test 2, model against Elo:", elo_test)

,method,accuracy_pct,log_loss
0,Final model,65.28,0.6243
1,Elo only,63.77,0.6343
2,ATP ranking only,62.49,0.6428


Test 2, model against Elo: {'gain': 0.01004, 'low_95': 1e-05, 'high_95': 0.02002, 'passes': True}


**Finding:** the model again beats Elo and the ATP ranking, and test 2 passes. But the lower end of the interval is only +0.00001, so a slightly different sample could fail it. The score is also well below notebook 08's 0.5929, and Elo fell from 0.6083 to 0.6343, so this period looks harder for both. The data can't tell us how much comes from the matches themselves and how much from the change of source and the missing Futures matches.

## 3. Against the bookmakers

In [3]:
names = names_for_odds_link("../data/raw", pd.read_sql_query("SELECT * FROM player_links", latest))
test = test.merge(names, on=KEYS, how="left", validate="one_to_one")
odds = load_odds("../data/raw/odds", first_year=2026)
odds = odds[odds["Date"] >= "2026-06-01"]
for column in ["BFEW", "BFEL"]:
    odds[column] = pd.to_numeric(odds[column], errors="coerce")
total = 1 / odds["BFEW"].where(odds["BFEW"] > 1) + 1 / odds["BFEL"].where(odds["BFEL"] > 1)
odds[["BFEW", "BFEL"]] = odds[["BFEW", "BFEL"]].where(total.between(0.95, 1.25))
test["odds_row"] = link_odds(test, odds)
linked = test.dropna(subset=["odds_row"]).join(odds, on="odds_row")
print(f"Linked {len(linked):,} of {len(test):,} matches. Without odds: Paris Masters, which was still being played, and the Laver Cup.")

rows = []
for label, w, l in [("Market average", "AvgW", "AvgL"), ("Bet365", "B365W", "B365L"), ("Betfair Exchange", "BFEW", "BFEL")]:
    part = linked.dropna(subset=[w, l])
    rows.append({"benchmark": label, "matches": len(part), "avg_margin_pct": round(100 * (1 / part[w] + 1 / part[l] - 1).mean(), 2),
                 "accuracy_pct": score(fair_probability(part[w], part[l]))["accuracy_pct"], "log_loss": score(fair_probability(part[w], part[l]))["log_loss"],
                 "model_accuracy_pct": score(part["model_p"])["accuracy_pct"], "model_log_loss": score(part["model_p"])["log_loss"]})
display(pd.DataFrame(rows))

market = linked.dropna(subset=["AvgW", "AvgL"])
market_test = interval(np.log(market["model_p"]) - np.log(fair_probability(market["AvgW"], market["AvgL"])))
print("Test 1, model against the market average:", market_test)

Linked 1,202 of 1,253 matches. Without odds: Paris Masters, which was still being played, and the Laver Cup.


,benchmark,matches,avg_margin_pct,accuracy_pct,log_loss,model_accuracy_pct,model_log_loss
0,Market average,1202,6.35,65.43,0.6153,65.39,0.6221
1,Bet365,1195,5.20,65.82,0.6127,65.52,0.6207
2,Betfair Exchange,1195,0.66,65.36,0.6121,65.52,0.6208


Test 1, model against the market average: {'gain': -0.00683, 'low_95': -0.01646, 'high_95': 0.00266, 'passes': False}


**Finding:** test 1 fails. Accuracy is about the same (65.39% against 65.43%), but the market's probabilities are better: its log loss is lower by 0.0068, and the interval includes zero. The gap is close to the 0.0070 between the model and Pinnacle in notebook 08, although that was a different bookmaker on different years. All three benchmarks score between 0.6121 and 0.6153, so the market also found this period harder.

## 4. The betting rule

As in notebook 08, a bet is made on at most one player per match, and retirements are refunded.

In [4]:
def simulate(frame, winner_col, loser_col, threshold=0.05):
    rows = frame.dropna(subset=[winner_col, loser_col]).copy()
    edge_winner = rows["model_p"] * rows[winner_col] - 1
    edge_loser = (1 - rows["model_p"]) * rows[loser_col] - 1
    on_winner = (edge_winner >= threshold) & (edge_winner >= edge_loser)
    on_loser = (edge_loser >= threshold) & (edge_loser > edge_winner)
    rows = rows[on_winner | on_loser].copy()
    won = on_winner[rows.index]
    rows["bet_odds"] = np.where(won, rows[winner_col], rows[loser_col])
    rows["profit"] = np.where(won, rows[winner_col] - 1, -1.0)
    return rows[~rows["Comment"].isin(["Retired", "Awarded"])]


def summarise(bets, label):
    summary = roi_summary(bets["profit"])
    return {"strategy": label, "bets": summary["bets"], "avg_odds": round(bets["bet_odds"].mean(), 2),
            "bets_won_pct": round(100 * (bets["profit"] > 0).mean(), 1), "profit_units": summary["profit_units"],
            "roi_pct": summary["roi_pct"], "low_95": summary["low_95"], "high_95": summary["high_95"]}


main_bets = simulate(linked, "B365W", "B365L")
betting = summarise(main_bets, "Test 3: Bet365, edge of 5% or more")
print("Test 3 passes:", betting["low_95"] > 0)
pd.DataFrame([betting, summarise(simulate(linked, "AvgW", "AvgL"), "Market average, edge of 5% or more"),
              summarise(simulate(linked, "MaxW", "MaxL"), "Best price, edge of 5% or more")])

Test 3 passes: False


,strategy,bets,avg_odds,bets_won_pct,profit_units,roi_pct,low_95,high_95
0,"Test 3: Bet365, edge of 5% or more",426,2.84,43.4,3.4,0.81,-11.66,13.91
1,"Market average, edge of 5% or more",361,2.77,43.5,7.6,2.09,-11.73,16.24
2,"Best price, edge of 5% or more",564,2.83,42.0,-14.7,-2.60,-13.79,9.00


**Finding:** test 3 fails. The return at Bet365 is +0.81%, but with 426 bets the interval runs from −11.66% to +13.91%, so the result says very little either way. The other prices disagree in direction, at +2.09% and −2.60%, which also suggests noise.

## 5. Are the probabilities still reliable?

The slope says how much the model's log-odds should be stretched. 1 is ideal, and below 1 means the model is too confident.

In [5]:
def calibration_slope(p):
    logit = np.log(p / (1 - p)).to_numpy()
    slopes = np.linspace(0.3, 1.7, 281)
    return float(slopes[np.argmax([-np.log1p(np.exp(-s * logit)).mean() for s in slopes])])


print(f"Calibration slope: model {calibration_slope(test['model_p']):.2f}, Elo {calibration_slope(test['elo_p']):.2f}")

Calibration slope: model 0.85, Elo 0.83


**Finding:** the model looks too confident here, with a slope of 0.85. Elo is similar at 0.83, which fits a period with more surprises. A correction fitted on these same months wouldn't be a fair test, so no change is made.

## 6. Exploratory: the seed rule without Pinnacle

**This isn't the frozen rule.** It takes the fair probability from the market average instead of Pinnacle, and bets at the best price, which may not have been available to bet. It only adds a little evidence for the lead in notebook 10.

In [6]:
rows = linked.dropna(subset=["AvgW", "AvgL", "MaxW", "MaxL", "d_seed"])
fair = fair_probability(rows["AvgW"], rows["AvgL"])
sides = pd.concat([
    pd.DataFrame({"p": seed_rule_probability(fair, rows["d_seed"]), "odds": rows["MaxW"], "won": 1.0, "comment": rows["Comment"]}),
    pd.DataFrame({"p": seed_rule_probability(1 - fair, -rows["d_seed"]), "odds": rows["MaxL"], "won": 0.0, "comment": rows["Comment"]}),
], ignore_index=True)
sides = sides[sides["comment"] == "Completed"].copy()
sides["profit"] = np.where(sides["won"] == 1, sides["odds"] - 1, -1.0)
print(roi_summary(sides.loc[sides["p"] * sides["odds"] - 1 >= 0.02, "profit"]))

{'bets': 209, 'profit_units': -13.0, 'roi_pct': -6.24, 'low_95': -27.82, 'high_95': 17.46}


**Finding:** on 209 bets, the return is −6.24% (−27.82% to +17.46%). The interval is too wide to settle anything, but this doesn't support the lead from notebook 10. Testing the real rule needs Pinnacle's prices, which the file doesn't have for this period.

## 7. Conclusions

| Question | Answer |
|---|---|
| Is the model better than Elo on new matches? | Yes under the rule, but only just |
| Is it better than the market average? | No: the market's log loss is lower, and the interval includes zero |
| Does the betting rule make money? | Not shown: +0.81%, with a very wide interval |
| Does the seed rule still work? | Unknown: its frozen form needs Pinnacle's prices, and the exploratory version doesn't support it |
| Should real money be used? | No |

**This test can't be repeated.** These matches have now been seen, so any model changed because of them needs newer matches for a clean test.

**Limits**
- **A small and short sample:** 1,253 matches from about five months.
- **A change of data source:** TennisMyLife replaces Sackmann in these months, with no Futures matches, and a few player links rely on name and age.
- **A different benchmark:** without Pinnacle, the market average carries a larger margin, though margin is removed before comparing probabilities.
- **Prices:** Paris Masters and the Laver Cup have no odds, and the file doesn't say when each price was offered.